### Advanced Coroutines, Queues and Cooperative Scheduling


In the original introduction we used generators to voluntarily suspend execution and hand control back to a coordinator.

In this notebook we are going to keep that same style, but take the ideas quite a bit further.

We will build the solutions gradually, and we will use small code cells so that every moving part can be inspected independently.

The main topics are:

* FIFO queues using `deque`
* bounded queues and explicit overflow policies
* producer and consumer coroutines
* sending values into a coroutine
* composing coroutine pipelines
* round-robin scheduling
* logical sleeping and timed tasks
* cancellation and exception injection
* rolling statistics
* correct queue benchmarks
* a complete resilient streaming project

All examples use the Python standard library only.


In [1]:
from collections import Counter, deque
from dataclasses import dataclass, field
from enum import Enum, auto
from functools import wraps
from typing import Any, Callable, Generator, Iterable, Iterator
import heapq
import inspect
import itertools
import math
import statistics
import timeit


Before solving the larger problems, let's create one small helper.

A coroutine that receives values using `send` has to be advanced to its first `yield` before it can receive a non-`None` value.


In [2]:
def coroutine(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        generator = function(*args, **kwargs)
        next(generator)
        return generator
    return wrapper


Let's verify the helper with a very small coroutine.


In [3]:
@coroutine
def echo():
    while True:
        value = yield
        print('received:', value)


In [4]:
receiver = echo()
receiver.send('hello')
receiver.send(100)
receiver.close()


received: hello
received: 100


### Problem 1 - Build a FIFO Queue Class


A queue is normally First-In First-Out.

We will use the convention:

* add new work on the right using `append`
* remove old work from the left using `popleft`

The goal is to wrap that behavior in a small class and add a few useful operations.


In [5]:
class FIFOQueue:
    def __init__(self, values=None):
        self._items = deque(values or [])

    def put(self, item):
        self._items.append(item)

    def get(self):
        if not self._items:
            raise IndexError('get from empty queue')
        return self._items.popleft()

    def peek(self):
        if not self._items:
            raise IndexError('peek from empty queue')
        return self._items[0]

    def __len__(self):
        return len(self._items)

    def __bool__(self):
        return bool(self._items)

    def snapshot(self):
        return tuple(self._items)


Let's try the basic operations.


In [6]:
queue = FIFOQueue([10, 20, 30])
queue.snapshot()


(10, 20, 30)

In [7]:
queue.put(40)
queue.snapshot()


(10, 20, 30, 40)

In [8]:
queue.peek()


10

In [9]:
[queue.get(), queue.get(), queue.get(), queue.get()]


[10, 20, 30, 40]

In [10]:
assert not queue


All of the important end operations are constant time.

The `snapshot` method is linear because it creates a copy.


### Problem 2 - A Bounded Queue with Explicit Overflow Rules


A `deque` can have a `maxlen`, but when it is full it silently discards an item.

That is excellent for rolling history, but it can be dangerous for a work queue.

For this problem we will support two explicit policies:

* reject the new item
* remove the oldest item and accept the new one

We will also keep metrics so that rejected or evicted data is visible.


In [11]:
class OverflowPolicy(Enum):
    REJECT_NEW = auto()
    DROP_OLDEST = auto()


In [12]:
@dataclass(frozen=True)
class PutResult:
    accepted: bool
    evicted: Any = None


In [13]:
class BoundedFIFO:
    def __init__(self, capacity, policy=OverflowPolicy.REJECT_NEW):
        if capacity <= 0:
            raise ValueError('capacity must be positive')

        self.capacity = capacity
        self.policy = policy
        self._items = deque()

        self.accepted_count = 0
        self.rejected_count = 0
        self.evicted_count = 0

    def __len__(self):
        return len(self._items)

    def __bool__(self):
        return bool(self._items)

    def is_full(self):
        return len(self) == self.capacity

    def put(self, item):
        if not self.is_full():
            self._items.append(item)
            self.accepted_count += 1
            return PutResult(True)

        if self.policy is OverflowPolicy.REJECT_NEW:
            self.rejected_count += 1
            return PutResult(False)

        evicted = self._items.popleft()
        self._items.append(item)
        self.accepted_count += 1
        self.evicted_count += 1
        return PutResult(True, evicted)

    def get(self):
        if not self._items:
            raise IndexError('get from empty bounded queue')
        return self._items.popleft()

    def snapshot(self):
        return tuple(self._items)


First let's use the reject-new policy.


In [14]:
rejecting = BoundedFIFO(3, OverflowPolicy.REJECT_NEW)

for value in [1, 2, 3, 4, 5]:
    print(value, rejecting.put(value))


1 PutResult(accepted=True, evicted=None)
2 PutResult(accepted=True, evicted=None)
3 PutResult(accepted=True, evicted=None)
4 PutResult(accepted=False, evicted=None)
5 PutResult(accepted=False, evicted=None)


In [15]:
rejecting.snapshot(), rejecting.accepted_count, rejecting.rejected_count


((1, 2, 3), 3, 2)

Now let's use the rolling policy.


In [16]:
rolling = BoundedFIFO(3, OverflowPolicy.DROP_OLDEST)

for value in ['A', 'B', 'C', 'D', 'E']:
    print(value, rolling.put(value), rolling.snapshot())


A PutResult(accepted=True, evicted=None) ('A',)
B PutResult(accepted=True, evicted=None) ('A', 'B')
C PutResult(accepted=True, evicted=None) ('A', 'B', 'C')
D PutResult(accepted=True, evicted='A') ('B', 'C', 'D')
E PutResult(accepted=True, evicted='B') ('C', 'D', 'E')


In [17]:
assert rolling.snapshot() == ('C', 'D', 'E')
assert rolling.evicted_count == 2


### Problem 3 - A Cooperative Producer and Consumer


We will now create a producer and a consumer that voluntarily yield control.

The producer must stop when the queue is full.

The consumer will process only a limited number of items in one turn. This is important because a consumer that drains a huge queue without yielding can block every other task.


In [18]:
@dataclass
class FlowMetrics:
    produced: int = 0
    consumed: int = 0
    producer_blocked: int = 0
    consumer_idle: int = 0
    maximum_queue_size: int = 0


In [19]:
def producer(values, queue, metrics):
    for value in values:
        while queue.is_full():
            metrics.producer_blocked += 1
            print('producer blocked - yielding control')
            yield 'producer blocked'

        result = queue.put(value)
        assert result.accepted

        metrics.produced += 1
        metrics.maximum_queue_size = max(
            metrics.maximum_queue_size,
            len(queue),
        )

        print('produced', value)
        yield 'produced'


In [20]:
def consumer(queue, output, metrics, producer_finished, batch_size=2):
    while not producer_finished() or queue:
        processed = 0

        while queue and processed < batch_size:
            value = queue.get()
            output.append(value)
            metrics.consumed += 1
            processed += 1
            print('consumed', value)

        if processed == 0:
            metrics.consumer_idle += 1
            print('consumer idle - yielding control')

        yield 'consumer turn complete'


The coordinator alternates between the two generators.

It also has to handle the fact that the producer may finish before the consumer.


In [21]:
def coordinate_flow(values, capacity=4, consumer_batch_size=2):
    queue = BoundedFIFO(capacity)
    output = []
    metrics = FlowMetrics()
    state = {'producer_finished': False}

    def producer_wrapper():
        yield from producer(values, queue, metrics)
        state['producer_finished'] = True

    producer_task = producer_wrapper()
    consumer_task = consumer(
        queue,
        output,
        metrics,
        producer_finished=lambda: state['producer_finished'],
        batch_size=consumer_batch_size,
    )

    tasks = deque([
        ('producer', producer_task),
        ('consumer', consumer_task),
    ])

    while tasks:
        name, task = tasks.popleft()

        try:
            next(task)
        except StopIteration:
            print(name, 'finished')
        else:
            tasks.append((name, task))

    return output, metrics


In [22]:
flow_output, flow_metrics = coordinate_flow(
    list(range(1, 13)),
    capacity=4,
    consumer_batch_size=2,
)


produced 1
consumed 1
produced 2
consumed 2
produced 3
consumed 3
produced 4
consumed 4
produced 5
consumed 5
produced 6
consumed 6
produced 7
consumed 7
produced 8
consumed 8
produced 9
consumed 9
produced 10
consumed 10
produced 11
consumed 11
produced 12
consumed 12
producer finished
consumer finished


In [23]:
flow_output, flow_metrics


([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
 FlowMetrics(produced=12, consumed=12, producer_blocked=0, consumer_idle=0, maximum_queue_size=1))

In [24]:
assert flow_output == list(range(1, 13))
assert flow_metrics.produced == 12
assert flow_metrics.consumed == 12
assert flow_metrics.maximum_queue_size <= 4


### Problem 4 - Send Values into a Running Statistics Coroutine


Coroutines can receive values through `send`.

For this problem we will keep count, mean, variance, minimum and maximum without storing the complete input history.

We will use Welford's algorithm for the variance.


In [25]:
@dataclass(frozen=True)
class StatisticsSnapshot:
    count: int
    mean: float
    population_variance: float
    minimum: Any
    maximum: Any


In [26]:
SNAPSHOT = object()
STOP = object()


In [27]:
def online_statistics():
    count = 0
    mean = 0.0
    m2 = 0.0
    minimum = None
    maximum = None

    message = yield None

    while True:
        if message is SNAPSHOT:
            variance = m2 / count if count else 0.0
            message = yield StatisticsSnapshot(
                count,
                mean,
                variance,
                minimum,
                maximum,
            )
            continue

        if message is STOP:
            variance = m2 / count if count else 0.0
            return StatisticsSnapshot(
                count,
                mean,
                variance,
                minimum,
                maximum,
            )

        value = float(message)
        count += 1

        delta = value - mean
        mean += delta / count
        delta_after_mean_change = value - mean
        m2 += delta * delta_after_mean_change

        minimum = value if minimum is None else min(minimum, value)
        maximum = value if maximum is None else max(maximum, value)

        message = yield None


This coroutine is not automatically primed because it returns an initial value from the first `yield`.


In [28]:
stats = online_statistics()
next(stats)


In [29]:
for value in [2, 4, 4, 4, 5, 5, 7, 9]:
    stats.send(value)


In [30]:
snapshot = stats.send(SNAPSHOT)
snapshot


StatisticsSnapshot(count=8, mean=5.0, population_variance=4.0, minimum=2.0, maximum=9.0)

In [31]:
assert snapshot.count == 8
assert math.isclose(snapshot.mean, 5.0)
assert math.isclose(snapshot.population_variance, 4.0)


In [32]:
try:
    stats.send(STOP)
except StopIteration as stop:
    final_snapshot = stop.value

final_snapshot


StatisticsSnapshot(count=8, mean=5.0, population_variance=4.0, minimum=2.0, maximum=9.0)

### Problem 5 - Build a Coroutine Processing Pipeline


We will now connect multiple coroutines together.

Our pipeline will:

1. receive numbers
2. square each number
3. keep only even squares
4. broadcast each remaining number to two output lists

Each stage will close its downstream stage when it is closed.


In [33]:
@coroutine
def collect(target):
    try:
        while True:
            value = yield
            target.append(value)
    finally:
        print('collector closed')


In [34]:
@coroutine
def transform(function, target):
    try:
        while True:
            value = yield
            target.send(function(value))
    finally:
        target.close()


In [35]:
@coroutine
def select(predicate, target):
    try:
        while True:
            value = yield
            if predicate(value):
                target.send(value)
    finally:
        target.close()


In [36]:
@coroutine
def broadcast(targets):
    active_targets = list(targets)

    try:
        while True:
            value = yield
            surviving_targets = []

            for target in active_targets:
                try:
                    target.send(value)
                except StopIteration:
                    pass
                else:
                    surviving_targets.append(target)

            active_targets = surviving_targets
    finally:
        for target in active_targets:
            target.close()


In [37]:
left = []
right = []

pipeline = transform(
    lambda value: value ** 2,
    select(
        lambda value: value % 2 == 0,
        broadcast([
            collect(left),
            collect(right),
        ]),
    ),
)


In [38]:
for value in range(1, 11):
    pipeline.send(value)

pipeline.close()


collector closed
collector closed


In [39]:
left, right


([4, 16, 36, 64, 100], [4, 16, 36, 64, 100])

In [40]:
assert left == [4, 16, 36, 64, 100]
assert right == left


### Problem 6 - A Fair Round-Robin Scheduler


A round-robin scheduler gives each ready task one turn and then places the unfinished task at the back of the queue.

We will also record return values and failures.


In [41]:
@dataclass
class TaskFailure:
    task_name: str
    error: BaseException


In [42]:
@dataclass
class SchedulerReport:
    trace: list = field(default_factory=list)
    completions: dict = field(default_factory=dict)
    failures: list = field(default_factory=list)


In [43]:
class RoundRobinScheduler:
    def __init__(self):
        self.ready = deque()
        self.report = SchedulerReport()

    def add(self, name, task):
        self.ready.append((name, task))

    def run(self, maximum_steps=None):
        steps = 0

        while self.ready:
            if maximum_steps is not None and steps >= maximum_steps:
                break

            name, task = self.ready.popleft()

            try:
                event = next(task)
            except StopIteration as stop:
                self.report.completions[name] = stop.value
            except BaseException as error:
                self.report.failures.append(TaskFailure(name, error))
            else:
                self.report.trace.append(event)
                self.ready.append((name, task))

            steps += 1

        return self.report


In [44]:
def counting_task(name, steps):
    for index in range(1, steps + 1):
        yield f'{name}:{index}'
    return f'{name} complete'


In [45]:
def broken_task():
    yield 'broken:before-error'
    raise RuntimeError('simulated task failure')


In [46]:
scheduler = RoundRobinScheduler()
scheduler.add('A', counting_task('A', 3))
scheduler.add('B', counting_task('B', 2))
scheduler.add('C', counting_task('C', 4))
scheduler.add('broken', broken_task())

scheduler_report = scheduler.run()
scheduler_report


SchedulerReport(trace=['A:1', 'B:1', 'C:1', 'broken:before-error', 'A:2', 'B:2', 'C:2', 'A:3', 'C:3', 'C:4'], completions={'B': 'B complete', 'A': 'A complete', 'C': 'C complete'}, failures=[TaskFailure(task_name='broken', error=RuntimeError('simulated task failure'))])

In [47]:
scheduler_report.trace


['A:1',
 'B:1',
 'C:1',
 'broken:before-error',
 'A:2',
 'B:2',
 'C:2',
 'A:3',
 'C:3',
 'C:4']

In [48]:
assert scheduler_report.trace == [
    'A:1',
    'B:1',
    'C:1',
    'broken:before-error',
    'A:2',
    'B:2',
    'C:2',
    'A:3',
    'C:3',
    'C:4',
]
assert len(scheduler_report.failures) == 1


The scheduler is fair only if every task yields reasonably quickly.

A task that performs a very long calculation before yielding still blocks every other task.


### Problem 7 - Add Logical Sleeping


Real event loops have to manage tasks that are temporarily not ready.

We will represent a sleeping task with a wake-up tick.

A `deque` will hold ready tasks and a heap will hold sleeping tasks.


In [49]:
@dataclass(frozen=True)
class YieldNow:
    pass


In [50]:
@dataclass(frozen=True)
class Sleep:
    ticks: int

    def __post_init__(self):
        if self.ticks < 0:
            raise ValueError('ticks cannot be negative')


In [51]:
@dataclass
class TimedTask:
    task_id: int
    name: str
    generator: Generator


In [52]:
@dataclass(frozen=True)
class TimedEvent:
    tick: int
    task_name: str
    command: Any


In [53]:
class TimedScheduler:
    def __init__(self):
        self.now = 0
        self.ready = deque()
        self.sleeping = []
        self.next_task_id = itertools.count(1)
        self.sequence = itertools.count()

        self.events = []
        self.completions = {}
        self.failures = {}

    def create_task(self, name, generator):
        task_id = next(self.next_task_id)
        self.ready.append(TimedTask(task_id, name, generator))
        return task_id

    def wake_due_tasks(self):
        while self.sleeping and self.sleeping[0][0] <= self.now:
            _, _, task = heapq.heappop(self.sleeping)
            self.ready.append(task)

    def run(self):
        while self.ready or self.sleeping:
            self.wake_due_tasks()

            if not self.ready:
                self.now = self.sleeping[0][0]
                self.wake_due_tasks()

            task = self.ready.popleft()

            try:
                command = next(task.generator)
            except StopIteration as stop:
                self.completions[task.name] = stop.value
                self.events.append(
                    TimedEvent(self.now, task.name, 'completed')
                )
                continue
            except BaseException as error:
                self.failures[task.name] = error
                self.events.append(
                    TimedEvent(
                        self.now,
                        task.name,
                        f'failed:{type(error).__name__}',
                    )
                )
                continue

            self.events.append(TimedEvent(self.now, task.name, command))

            if isinstance(command, YieldNow):
                self.ready.append(task)
            elif isinstance(command, Sleep):
                wake_tick = self.now + command.ticks
                heapq.heappush(
                    self.sleeping,
                    (wake_tick, next(self.sequence), task),
                )
            else:
                self.failures[task.name] = TypeError(
                    f'unsupported scheduler command: {command!r}'
                )

            self.now += 1


In [54]:
def periodic_task(name, repeats, delay):
    for index in range(1, repeats + 1):
        print(name, 'cycle', index, 'sleeping for', delay)
        yield Sleep(delay)
        print(name, 'awake')
        yield YieldNow()

    return f'{name} finished'


In [55]:
timed_scheduler = TimedScheduler()
timed_scheduler.create_task('fast', periodic_task('fast', 3, 1))
timed_scheduler.create_task('slow', periodic_task('slow', 2, 4))
timed_scheduler.run()


fast cycle 1 sleeping for 1
slow cycle 1 sleeping for 4
fast awake
fast cycle 2 sleeping for 1
fast awake
fast cycle 3 sleeping for 1
slow awake
fast awake
slow cycle 2 sleeping for 4
slow awake


In [56]:
timed_scheduler.events


[TimedEvent(tick=0, task_name='fast', command=Sleep(ticks=1)),
 TimedEvent(tick=1, task_name='slow', command=Sleep(ticks=4)),
 TimedEvent(tick=2, task_name='fast', command=YieldNow()),
 TimedEvent(tick=3, task_name='fast', command=Sleep(ticks=1)),
 TimedEvent(tick=4, task_name='fast', command=YieldNow()),
 TimedEvent(tick=5, task_name='fast', command=Sleep(ticks=1)),
 TimedEvent(tick=6, task_name='slow', command=YieldNow()),
 TimedEvent(tick=7, task_name='fast', command=YieldNow()),
 TimedEvent(tick=8, task_name='slow', command=Sleep(ticks=4)),
 TimedEvent(tick=9, task_name='fast', command='completed'),
 TimedEvent(tick=12, task_name='slow', command=YieldNow()),
 TimedEvent(tick=13, task_name='slow', command='completed')]

In [57]:
assert timed_scheduler.completions == {
    'fast': 'fast finished',
    'slow': 'slow finished',
}
assert not timed_scheduler.failures


### Problem 8 - Cancellation and Exception Injection


A cooperative task can only observe cancellation when it gives control back to the scheduler.

We will request cancellation externally and inject a custom exception at the next step.


In [58]:
class TaskCancelled(Exception):
    pass


In [59]:
@dataclass
class CancellableTask:
    generator: Generator
    cancellation_requested: bool = False

    def cancel(self):
        self.cancellation_requested = True

    def step(self):
        if self.cancellation_requested:
            self.cancellation_requested = False
            return self.generator.throw(TaskCancelled())

        return next(self.generator)


In [60]:
def cancellable_worker(lifecycle):
    lifecycle.append('started')

    try:
        while True:
            lifecycle.append('working')
            yield YieldNow()
    except TaskCancelled:
        lifecycle.append('cancellation observed')
        return 'cancelled'
    finally:
        lifecycle.append('cleaned up')


In [61]:
cancellation_lifecycle = []
cancellable = CancellableTask(
    cancellable_worker(cancellation_lifecycle)
)


In [62]:
cancellable.step()
cancellable.step()
cancellable.cancel()


In [63]:
try:
    cancellable.step()
except StopIteration as stop:
    cancellation_result = stop.value

cancellation_result, cancellation_lifecycle


('cancelled',
 ['started', 'working', 'working', 'cancellation observed', 'cleaned up'])

In [64]:
assert cancellation_result == 'cancelled'
assert cancellation_lifecycle[-2:] == [
    'cancellation observed',
    'cleaned up',
]


### Problem 9 - A Rolling Average and Anomaly Detector


This time `deque(maxlen=...)` is exactly what we want.

The queue represents a rolling history, so automatically removing the oldest value is part of the design.

First we will implement an O(1) moving average.


In [65]:
class MovingAverage:
    def __init__(self, window_size):
        if window_size <= 0:
            raise ValueError('window_size must be positive')

        self.window = deque(maxlen=window_size)
        self.total = 0.0

    def add(self, value):
        value = float(value)

        if len(self.window) == self.window.maxlen:
            self.total -= self.window[0]

        self.window.append(value)
        self.total += value

        return self.total / len(self.window)


In [66]:
averager = MovingAverage(3)

averages = [averager.add(value) for value in [1, 2, 3, 4, 5]]
averages


[1.0, 1.5, 2.0, 3.0, 4.0]

In [67]:
assert averages == [1.0, 1.5, 2.0, 3.0, 4.0]


Now let's build a coroutine that detects values far from the recent baseline.


In [68]:
@dataclass(frozen=True)
class Anomaly:
    value: float
    mean: float
    standard_deviation: float
    z_score: float


In [69]:
@coroutine
def anomaly_detector(window_size, z_threshold, output):
    if window_size < 2:
        raise ValueError('window_size must be at least 2')

    history = deque(maxlen=window_size)

    while True:
        value = float((yield))

        if len(history) < window_size:
            history.append(value)
            continue

        mean = statistics.fmean(history)
        standard_deviation = statistics.pstdev(history)

        if standard_deviation == 0:
            z_score = math.inf if value != mean else 0.0
        else:
            z_score = abs(value - mean) / standard_deviation

        if z_score > z_threshold:
            output.append(
                Anomaly(value, mean, standard_deviation, z_score)
            )
        else:
            history.append(value)


In [70]:
anomalies = []
detector = anomaly_detector(5, 3.0, anomalies)

for value in [10.0, 10.1, 9.9, 10.2, 9.8, 10.0, 50.0, 10.1]:
    detector.send(value)

detector_result = None


In [71]:
detector.close()
anomalies


[Anomaly(value=50.0, mean=10.0, standard_deviation=0.141421356237309, z_score=282.84271247462)]

In [72]:
assert len(anomalies) == 1
assert anomalies[0].value == 50.0


### Problem 10 - Correct Queue Benchmarks


A benchmark should create a fresh mutable collection for every timed invocation.

If we reuse a global collection and empty it during the first call, later repetitions no longer measure the same operation.


In [73]:
def list_append_right(n):
    values = []
    for value in range(n):
        values.append(value)


In [74]:
def deque_append_right(n):
    values = deque()
    for value in range(n):
        values.append(value)


In [75]:
def list_insert_left(n):
    values = []
    for value in range(n):
        values.insert(0, value)


In [76]:
def deque_append_left(n):
    values = deque()
    for value in range(n):
        values.appendleft(value)


In [77]:
def list_pop_left(n):
    values = list(range(n))
    while values:
        values.pop(0)


In [78]:
def deque_pop_left(n):
    values = deque(range(n))
    while values:
        values.popleft()


In [79]:
def benchmark_queues(n=2_000, repeats=3):
    functions = {
        'list append right': list_append_right,
        'deque append right': deque_append_right,
        'list insert left': list_insert_left,
        'deque append left': deque_append_left,
        'list pop left': list_pop_left,
        'deque pop left': deque_pop_left,
    }

    results = {}

    for name, function in functions.items():
        timer = timeit.Timer(lambda fn=function: fn(n))
        results[name] = min(timer.repeat(repeat=repeats, number=1))

    return results


In [80]:
benchmark_results = benchmark_queues()

for name, seconds in sorted(
    benchmark_results.items(),
    key=lambda item: item[1],
):
    print(f'{name:<20} {seconds:.6f} seconds')


deque append left    0.000107 seconds
deque append right   0.000110 seconds
list append right    0.000117 seconds
deque pop left       0.000142 seconds
list insert left     0.001190 seconds
list pop left        0.005676 seconds


In [81]:
assert benchmark_results['deque append left'] < benchmark_results['list insert left']
assert benchmark_results['deque pop left'] < benchmark_results['list pop left']


### Final Project - A Resilient Cooperative Streaming Service


We will finish with a complete project.

The service has three cooperative stages:

1. a producer reads raw records
2. a transformer validates and normalizes them
3. a writer stores successful results

The system also supports:

* bounded input and output queues
* explicit backpressure
* transient retries
* permanent validation failures
* dead-letter storage
* bounded work per turn
* shutdown invariants

We will keep the stages as normal functions that perform one turn of work. The coordinator decides when each stage runs.


In [82]:
class PermanentRecordError(ValueError):
    pass


class TransientRecordError(RuntimeError):
    pass


In [83]:
@dataclass
class Envelope:
    record: dict
    attempt: int = 1


In [84]:
@dataclass
class ProjectMetrics:
    produced: int = 0
    transformed: int = 0
    written: int = 0
    retries: int = 0
    dead_letters: int = 0
    blocked_input: int = 0
    blocked_output: int = 0
    coordinator_turns: int = 0


In [85]:
@dataclass
class ProjectState:
    source: deque
    input_queue: BoundedFIFO
    output_queue: BoundedFIFO
    retry_queue: deque
    pending_normalized: deque
    written: list
    dead_letters: list
    transient_failures_remaining: dict
    metrics: ProjectMetrics = field(default_factory=ProjectMetrics)


The normalization function contains the business rules.

Some record IDs can be configured to fail temporarily a certain number of times.


In [86]:
def normalize_record(record, transient_failures_remaining):
    if 'id' not in record:
        raise PermanentRecordError('missing id')

    if 'name' not in record:
        raise PermanentRecordError('missing name')

    record_id = record['id']

    if not isinstance(record_id, int) or record_id <= 0:
        raise PermanentRecordError('id must be a positive integer')

    failures_left = transient_failures_remaining.get(record_id, 0)

    if failures_left > 0:
        transient_failures_remaining[record_id] = failures_left - 1
        raise TransientRecordError('temporary dependency failure')

    name = str(record['name']).strip()

    if not name:
        raise PermanentRecordError('name cannot be blank')

    return {
        'id': record_id,
        'name': name.title(),
    }


The producer fills available input capacity, but it performs only a limited amount of work in one turn.


In [87]:
def producer_turn(state, budget):
    progress = False

    for _ in range(budget):
        if not state.source:
            break

        if state.input_queue.is_full():
            state.metrics.blocked_input += 1
            break

        record = state.source.popleft()
        result = state.input_queue.put(Envelope(record))
        assert result.accepted

        state.metrics.produced += 1
        progress = True

    return progress


The transformer gives retry work priority.

A record that has already been normalized but cannot enter a full output queue is stored separately. This is clearer than placing hidden marker fields inside the business record.


In [88]:
def transformer_turn(state, budget, maximum_attempts):
    progress = False

    for _ in range(budget):
        if state.pending_normalized:
            if state.output_queue.is_full():
                state.metrics.blocked_output += 1
                break

            normalized = state.pending_normalized.popleft()
            assert state.output_queue.put(normalized).accepted
            state.metrics.transformed += 1
            progress = True
            continue

        if state.retry_queue:
            envelope = state.retry_queue.popleft()
        elif state.input_queue:
            envelope = state.input_queue.get()
        else:
            break

        try:
            normalized = normalize_record(
                envelope.record,
                state.transient_failures_remaining,
            )
        except PermanentRecordError as error:
            state.dead_letters.append((envelope.record, str(error)))
            state.metrics.dead_letters += 1
            progress = True
            continue
        except TransientRecordError as error:
            if envelope.attempt < maximum_attempts:
                state.retry_queue.append(
                    Envelope(
                        envelope.record,
                        envelope.attempt + 1,
                    )
                )
                state.metrics.retries += 1
            else:
                state.dead_letters.append(
                    (
                        envelope.record,
                        f'retry exhausted: {error}',
                    )
                )
                state.metrics.dead_letters += 1

            progress = True
            continue

        if state.output_queue.is_full():
            state.pending_normalized.append(normalized)
            state.metrics.blocked_output += 1
            progress = True
            break

        assert state.output_queue.put(normalized).accepted
        state.metrics.transformed += 1
        progress = True

    return progress


The writer drains a limited number of records in each turn.


In [89]:
def writer_turn(state, budget):
    progress = False

    for _ in range(budget):
        if not state.output_queue:
            break

        state.written.append(state.output_queue.get())
        state.metrics.written += 1
        progress = True

    return progress


Now we can build the coordinator.

If work remains but none of the stages can make progress, the system is deadlocked and should fail loudly.


In [90]:
def run_streaming_project(
    records,
    input_capacity=3,
    output_capacity=2,
    producer_budget=2,
    transformer_budget=2,
    writer_budget=1,
    maximum_attempts=3,
    transient_failures=None,
):
    state = ProjectState(
        source=deque(records),
        input_queue=BoundedFIFO(input_capacity),
        output_queue=BoundedFIFO(output_capacity),
        retry_queue=deque(),
        pending_normalized=deque(),
        written=[],
        dead_letters=[],
        transient_failures_remaining=dict(transient_failures or {}),
    )

    while (
        state.source
        or state.input_queue
        or state.retry_queue
        or state.pending_normalized
        or state.output_queue
    ):
        state.metrics.coordinator_turns += 1

        progress = False
        progress |= producer_turn(state, producer_budget)
        progress |= transformer_turn(
            state,
            transformer_budget,
            maximum_attempts,
        )
        progress |= writer_turn(state, writer_budget)

        if not progress:
            raise RuntimeError(
                'unfinished work remains, but no stage can make progress'
            )

    return state


Let's run the complete project with successful, invalid and temporarily failing records.


In [91]:
records = [
    {'id': 1, 'name': '  ada lovelace '},
    {'id': 2, 'name': 'grace hopper'},
    {'id': -3, 'name': 'invalid id'},
    {'name': 'missing id'},
    {'id': 4, 'name': '   '},
    {'id': 5, 'name': 'alan turing'},
    {'id': 6, 'name': 'katherine johnson'},
]


In [92]:
project = run_streaming_project(
    records,
    transient_failures={
        2: 1,
        5: 5,
    },
)


In [93]:
project.written


[{'id': 1, 'name': 'Ada Lovelace'},
 {'id': 2, 'name': 'Grace Hopper'},
 {'id': 6, 'name': 'Katherine Johnson'}]

In [94]:
project.dead_letters


[({'id': -3, 'name': 'invalid id'}, 'id must be a positive integer'),
 ({'name': 'missing id'}, 'missing id'),
 ({'id': 4, 'name': '   '}, 'name cannot be blank'),
 ({'id': 5, 'name': 'alan turing'},
  'retry exhausted: temporary dependency failure')]

In [95]:
project.metrics


ProjectMetrics(produced=7, transformed=3, written=3, retries=3, dead_letters=4, blocked_input=0, blocked_output=0, coordinator_turns=5)

Finally, let's verify the important invariants.


In [96]:
assert project.written == [
    {'id': 1, 'name': 'Ada Lovelace'},
    {'id': 2, 'name': 'Grace Hopper'},
    {'id': 6, 'name': 'Katherine Johnson'},
]

assert len(project.dead_letters) == 4
assert project.metrics.produced == len(records)
assert project.metrics.produced == (
    project.metrics.written + project.metrics.dead_letters
)
assert project.metrics.transformed == project.metrics.written
assert not project.input_queue
assert not project.output_queue
assert not project.retry_queue
assert not project.pending_normalized


### Additional Solved Example - Batching Coroutine


A batching coroutine collects values and forwards complete tuples.

When it is closed, it can optionally flush the final partial batch.


In [97]:
@coroutine
def batcher(size, target, flush_partial=True):
    if size <= 0:
        raise ValueError('size must be positive')

    batch = []

    try:
        while True:
            value = yield
            batch.append(value)

            if len(batch) == size:
                target.send(tuple(batch))
                batch.clear()
    finally:
        try:
            if batch and flush_partial:
                target.send(tuple(batch))
        finally:
            target.close()


In [98]:
batches = []
batch_pipeline = batcher(3, collect(batches))

for value in range(8):
    batch_pipeline.send(value)

batch_pipeline.close()
batches


collector closed


[(0, 1, 2), (3, 4, 5), (6, 7)]

In [99]:
assert batches == [
    (0, 1, 2),
    (3, 4, 5),
    (6, 7),
]


### Additional Solved Example - Round-Robin Flattening


This generator takes one item from each iterable in turn.

When an iterator is exhausted, it is removed from the active queue.


In [100]:
def round_robin(*iterables):
    active = deque(map(iter, iterables))

    while active:
        iterator = active.popleft()

        try:
            value = next(iterator)
        except StopIteration:
            continue
        else:
            yield value
            active.append(iterator)


In [101]:
flattened = list(
    round_robin(
        [1, 2, 3],
        ['A', 'B'],
        [True, False, True, False],
    )
)

flattened


[1, 'A', True, 2, 'B', False, 3, True, False]

In [102]:
assert flattened == [
    1, 'A', True,
    2, 'B', False,
    3, True,
    False,
]


### Final Review


The most important ideas from these projects are:

* use `deque` for efficient operations at both ends
* use an explicit overflow policy for work queues
* yield regularly so cooperative tasks remain responsive
* use `send` to feed data into coroutine state machines
* use `finally` to close downstream stages and release resources
* do not swallow `GeneratorExit`
* isolate failures at task boundaries
* represent sleeping tasks separately from ready tasks
* treat cancellation as something observed at a yield point
* use rolling `deque(maxlen=...)` only when eviction is intentional
* test invariants as well as example outputs
* create fresh fixtures in microbenchmarks

For production asynchronous network applications, these concepts map naturally to `async def`, `await`, `asyncio.Queue`, tasks and event loops.


### Further Practice Problems


1. Add task priorities to the timed scheduler.
2. Add a `WaitTask(task_id)` command.
3. Add exponential retry delays to the streaming project.
4. Add graceful shutdown with a logical deadline.
5. Add per-task execution and waiting-time metrics.
6. Convert the producer-consumer example to `asyncio.Queue`.
7. Add a bounded channel with separate sender and receiver classes.
8. Add a timeout that injects an exception into a task.
9. Add random operation tests for `BoundedFIFO`.
10. Compare cooperative tasks with threads for I/O-bound work.
